# 4 · Linearization and the classical modes

Numerical and analytic linear models, the five modes, and the ten published 747 conditions.
Lesson: [04](../docs/lessons/04-linearization-and-modes.md) ·
Theory: [linearization and modes](../docs/linearization_and_modes.md).

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from flightdyn import plotting as P

P.use_style()
np.set_printoptions(precision=4, suppress=True)
sys.path.insert(0, "../validation")  # cited reference data and reference aircraft
from reference_aircraft import b747, b747_condition, navion, navion_condition
from flightdyn.linearization import analytic_model, lateral, linearize, longitudinal
from flightdyn.modal_analysis import lateral_modes, longitudinal_modes
from flightdyn.trim_solver import trim
from mode_validation import b747_table_rows, KEYS

In [2]:
ac, c = b747(2), b747_condition(2)
t = trim(ac, c["V"], c["h"])
lm = linearize(ac, t)
lon, lat = analytic_model(ac, t)
print("numerical vs analytic, largest difference:",
      f"{np.abs(longitudinal(lm).A - lon.A).max():.1e},",
      f"{np.abs(lateral(lm, c['V']).A - lat.A).max():.1e}")
modes = longitudinal_modes(lon.A) | lateral_modes(lat.A)
for name, m in modes.items():
    if m.is_oscillatory:
        print(f"{name:13s} wn {m.wn:.3f} rad/s  zeta {m.zeta:.3f}  period {m.period:.1f} s")
    else:
        print(f"{name:13s} T {m.time_constant:.2f} s")

numerical vs analytic, largest difference: 1.8e-10, 1.1e-10
phugoid       wn 0.126 rad/s  zeta 0.022  period 49.8 s
short_period  wn 0.908 rad/s  zeta 0.629  period 8.9 s
spiral        T 21.43 s
roll          T 0.82 s
dutch_roll    wn 0.745 rad/s  zeta 0.106  period 8.5 s


## The 747 at ten flight conditions, from the report's own derivatives

In [3]:
rows = b747_table_rows()
print("fc  " + "  ".join(f"{k:>12s}" for k in KEYS))
for r in rows:
    print(f"{r['fc']:2d}  " + "  ".join(
        f"{100 * (r['model'][k] / r['published'][k] - 1):+11.1f}%" for k in KEYS))

fc       ZETA_PH      OMEGA_PH       ZETA_SP      OMEGA_SP  INV_T_SPIRAL    INV_T_ROLL       ZETA_DR      OMEGA_DR
 1         -0.4%         -0.3%         +0.2%         -0.1%         -0.6%         -0.1%         +0.3%         -0.1%
 2         -1.1%         -0.3%         +0.0%         -0.1%         +0.5%         -0.6%         -0.7%         +0.1%
 3         -0.0%         +0.0%         +0.1%         -0.4%         +0.1%         -0.6%         +0.1%         -0.0%
 4         +0.0%         +0.1%         +0.0%         +0.4%         -0.2%         -0.0%         +0.3%         +0.4%
 5         -0.2%         -0.0%         +0.0%         -0.3%         -1.8%         +0.1%         +0.3%         -0.0%
 6         +0.3%         -0.1%         -0.2%         -0.3%         +0.4%         +0.0%         +0.1%         +0.4%
 7         -6.1%         +4.0%         -0.0%         +0.3%         +0.4%         -0.1%         +0.3%         -0.2%
 8         +0.2%         +0.0%         -0.0%         +0.1%         -0.7%        